
# Lab 01 - Sentiment Analysis Using Traditional Machine Learning





In [2]:
#Download and load the dataset.
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import pandas as pd
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

In [8]:
dataset_path = "../data/"
amazon = pd.read_csv(dataset_path + "amazon_cells_labelled.txt",sep="\t", names=["review", "sentiment"])

imdb  = pd.read_csv(dataset_path + "imdb_labelled.txt", sep="\t",names=["review", "sentiment"])

yelp= pd.read_csv(dataset_path + "yelp_labelled.txt", sep="\t", names=["review", "sentiment"])

In [10]:
print("Amazon dataset size:", amazon.shape)
print("IMDb dataset:", imdb.shape)
print("Yelp dataset:", yelp.shape)

Amazon dataset size: (1000, 2)
IMDb dataset: (748, 2)
Yelp dataset: (1000, 2)


In [11]:
data = pd.concat([amazon, imdb, yelp], ignore_index=True)
data.head()

,review,sentiment
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [12]:
print("Total reviews:", len(data))
print("Dataset shape:", data.shape)

Total reviews: 2748
Dataset shape: (2748, 2)


In [14]:
# basic text preprocessing
# Convert text to lowercase

data["clean_text"] = data["review"].str.lower()
data[["review", "clean_text"]].head()

,review,clean_text
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.","good case, excellent value."
2,Great for the jawbone.,great for the jawbone.
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great.


In [16]:
#Remove unnecessary characters and punctuation
data["clean_text"] = data["clean_text"].apply(lambda text: re.sub(r"[^a-zA-Z0-9\s]", " ", text))
data[["review", "clean_text"]].head()


,review,clean_text
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.",good case excellent value
2,Great for the jawbone.,great for the jawbone
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great


In [17]:
# Remove extra whitespace
def clean_spaces(text):return " ".join(text.split())
data["clean_text"] = data["clean_text"].apply(clean_spaces)

In [19]:
# Compare original and preprocessed text
data[["review", "clean_text"]].head(10)

,review,clean_text
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.",good case excellent value
2,Great for the jawbone.,great for the jawbone
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great
5,I have to jiggle the plug to get it to line up...,i have to jiggle the plug to get it to line up...
6,If you have several dozen or several hundred c...,if you have several dozen or several hundred c...
7,If you are Razr owner...you must have this!,if you are razr owner you must have this
8,"Needless to say, I wasted my money.",needless to say i wasted my money
9,What a waste of money and time!.,what a waste of money and time


In [22]:
#Explore the text data
#Display positive and negative examples
positive_reviews = data[data["sentiment"] == 1]
positive_reviews[["review", "sentiment"]].head()

negative_reviews = data[data["sentiment"] == 0]
negative_reviews[["review", "sentiment"]].head()

,review,sentiment
0,So there is no way for me to plug it in here i...,0
3,Tied to charger for conversations lasting more...,0
5,I have to jiggle the plug to get it to line up...,0
6,If you have several dozen or several hundred c...,0
8,"Needless to say, I wasted my money.",0


In [24]:
#Identify frequently occurring words
all_words = " ".join(data["clean_text"]).split()
word_counts = Counter(all_words)
common_words = word_counts.most_common(10)
print (common_words)


[('the', 1953), ('and', 1138), ('i', 1037), ('a', 889), ('it', 789), ('is', 754), ('to', 670), ('this', 643), ('of', 624), ('was', 571)]


In [25]:
# compare differences between positive and negative reviews
positive_words = " ".join(positive_reviews["clean_text"]).split()
negative_words = " ".join(negative_reviews["clean_text"]).split()

positive_counts = Counter(positive_words)
negative_counts = Counter(negative_words)

print("Most common words in positive reviews:")
print(positive_counts.most_common(10))

print("\nMost common words in negative reviews:")
print(negative_counts.most_common(10))


Most common words in positive reviews:
[('the', 960), ('and', 665), ('i', 475), ('a', 449), ('is', 397), ('it', 363), ('this', 318), ('to', 288), ('of', 287), ('was', 248)]

Most common words in negative reviews:
[('the', 993), ('i', 562), ('and', 473), ('a', 440), ('it', 426), ('to', 382), ('is', 357), ('of', 337), ('this', 325), ('was', 323)]


In [26]:
#devide the text and sentiment labels
X = data["clean_text"]
y = data["sentiment"]
print("Review data:")
print(X.head())

print("\nSentiment labels:")
print(y.head())

Review data:
0    so there is no way for me to plug it in here i...
1                            good case excellent value
2                                great for the jawbone
3    tied to charger for conversations lasting more...
4                                     the mic is great
Name: clean_text, dtype: str

Sentiment labels:
0    0
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

Training data: 2198
Testing data: 550


In [31]:
#Convert Text into Bag of Words (BOW)
#Convert reviews into numerical data
bow_convertor = CountVectorizer()

X_train_bow = bow_convertor.fit_transform(X_train)
X_test_bow = bow_convertor.transform(X_test)

print("Training data shape:", X_train_bow.shape)
print("Testing data shape:", X_test_bow.shape)

Training data shape: (2198, 4574)
Testing data shape: (550, 4574)


In [32]:
print("First review:")
print(X_train.iloc[0])

print("\nNumerical representation:")
print(X_train_bow[0].toarray())

First review:
i ve seen soap operas more intelligent than this movie

Numerical representation:
[[0 0 0 ... 0 0 0]]


In [34]:
#Train a ML model 
model = LogisticRegression(max_iter=1000)
model.fit(X_train_bow, y_train)
print("Model training completed succesfully!")


Model training completed succesfully!
